In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.filterwarnings("ignore")

In [2]:
df = sns.load_dataset("tips")
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [3]:
df.shape

(244, 7)

In [4]:
# predict what is the time ? if anyone is visiting-- is it linch or dinner ?? time is the target variable 
df.time.value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [5]:
df.time.isnull().sum()

np.int64(0)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [7]:
df.smoker.value_counts()

smoker
No     151
Yes     93
Name: count, dtype: int64

In [8]:
df.sex.value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [9]:
df.day.value_counts()

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

In [10]:
# target variable encoding : 
# df["time"] = df["time"].map({
#     "Lunch": 0,
#     "Dinner": 1
# })

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["time"] = le.fit_transform(df["time"])
print(le.classes_)  # print values in sorted order 

['Dinner' 'Lunch']


In [11]:
df.time.value_counts()
# 0 -> dinner , 1->lunch 

time
0    176
1     68
Name: count, dtype: int64

In [12]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4


In [13]:
X = df.drop("time", axis = 1)
y = df["time"]

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.2,random_state=1, stratify=y, shuffle=True)

In [15]:
print(type(X_train))

<class 'pandas.DataFrame'>


In [16]:
# handling missing value 
#encoding 
#scaling 

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


from sklearn.pipeline import Pipeline # a sequecne of data transformtion
from sklearn.compose import ColumnTransformer # group all the pipeline steps for each of the column 

In [17]:
df.dtypes

total_bill     float64
tip            float64
sex           category
smoker        category
day           category
time             int64
size             int64
dtype: object

In [18]:
cat_col = ["sex", "smoker", "day"]
num_col = ["total_bill", "tip", "size"]

In [19]:
#feature engineering automation using pipeline and column transformer 
num_pipeline = Pipeline(steps= [('imputation',SimpleImputer(strategy='median')),
                                ('scaling', StandardScaler())])

cat_pipeline = Pipeline(steps=[('imputation', SimpleImputer(strategy='most_frequent')),
                               ('endoding',OneHotEncoder())])

In [20]:
preprocessor = ColumnTransformer([("num_pipeline",num_pipeline,num_col),
                   ("cat_pipeline",cat_pipeline,cat_col)])

In [21]:
print(type(X_train))

<class 'pandas.DataFrame'>


In [22]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

In [24]:
lr = LogisticRegression(random_state=1)

dt = DecisionTreeClassifier(random_state=1)

svc = SVC(random_state=1)

gnb = GaussianNB()

models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Support Vector Machine": svc,
    "Gaussian Naive Bayes": gnb
}



In [25]:
# train and evalualte 
from sklearn.metrics import accuracy_score

for model_name, model in models.items():

    # Train
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    print(f"{model_name} : {accuracy:.4f}")

Logistic Regression : 0.9133
Decision Tree : 0.9592
Support Vector Machine : 0.9133
Gaussian Naive Bayes : 0.9592
